# Baseline: Perch embeddings

The baseline turns each 5-second segment into a Perch embedding, a compact numerical fingerprint of the sound, then trains a light classifier on top of those. Perch is a multi-taxa bioacoustics model, so its embeddings already carry meaning about birds, frogs, insects and mammals, which suits this dataset better than a bird-only model.

This first cell loads the segments prepared earlier and checks Perch produces the embedding shape expected before running it across everything.

In [6]:
import numpy as np
import json

# the raw waveforms, not the spectrograms: Perch takes audio, not images
# we saved spectrograms before, so we reload the segment audio here
specs = np.load('../data/full_specs.npy')
with open('../data/full_meta.json') as f:
    meta = json.load(f)

labels = meta['labels']
sites = meta['sites']
print("segments:", len(labels))
print("spec shape:", specs.shape)

segments: 1478
spec shape: (1478, 128, 313)


## Cutting the waveform segments

Perch takes raw 5-second audio, not spectrograms, so the segments are re-cut from the soundscape files as waveforms. The order matches the labels and sites already loaded, so nothing needs re-aligning afterwards.

In [7]:
import os
import re
import pandas as pd
import librosa

SR = 32000
DATA_DIR = '../data'

ss = pd.read_csv('../data/train_soundscapes_labels.csv')

def time_to_seconds(t):
    h, m, s = t.split(':')
    return int(h) * 3600 + int(m) * 60 + int(s)

# process files in the same order as before so waveforms line up with labels/sites
files = ss['filename'].unique()
files = [f for f in files if os.path.exists(os.path.join(DATA_DIR, f))]

waveforms = []
for n, fname in enumerate(files, 1):
    y_full, _ = librosa.load(os.path.join(DATA_DIR, fname), sr=SR)
    rows = ss[ss['filename'] == fname]
    for _, row in rows.iterrows():
        start = time_to_seconds(row['start'])
        end = time_to_seconds(row['end'])
        seg = y_full[start * SR : end * SR]
        target = SR * (end - start)
        if len(seg) < target:
            seg = np.pad(seg, (0, target - len(seg)))
        waveforms.append(seg.astype(np.float32))
    if n % 10 == 0:
        print(f"{n}/{len(files)} files")

waveforms = np.array(waveforms)
print("waveform stack:", waveforms.shape)

10/66 files
20/66 files
30/66 files
40/66 files
50/66 files
60/66 files
waveform stack: (1478, 160000)


## Loading Perch and embedding one segment

Perch is loaded once, then run on a single segment first to confirm it produces the expected 1536-dimensional embedding before processing all 1,478.

In [ ]:
# Load the Perch model and embed the first waveform
from perch_hoplite.zoo import model_configs

model = model_configs.load_model_by_name('perch_v2')

out = model.embed(waveforms[0])
print("embedding shape:", out.embeddings.shape)

c:\dev\biodiversity\mapping-biodiversity-from-sound\.venv\Lib\site-packages\perch_hoplite\zoo\zoo_interface.py:159: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  framed_audio = np.divide(framed_audio, peak_norm, where=(peak_norm > 0.0))


embedding shape: (1, 1, 1536)


## Embedding all segments

Every segment is passed through Perch, and the two singleton axes are squeezed off so each becomes a flat 1536-dimensional vector. The result is a 1478 by 1536 matrix, one row per segment, ready for a light classifier.

In [ ]:
import os

# cache the embeddings so we don't have to recompute them every time
emb_path = '../data/soundscape_embeddings.npy'
if os.path.exists(emb_path):
    embeddings = np.load(emb_path)
    print("loaded cached embeddings:", embeddings.shape)
else:
    embeddings = np.zeros((len(waveforms), 1536), dtype=np.float32)
    for i, wav in enumerate(waveforms):
        out = model.embed(wav)
        embeddings[i] = np.asarray(out.embeddings).reshape(-1)
        if (i + 1) % 100 == 0:
            print(f"{i+1}/{len(waveforms)} embedded")
    np.save(emb_path, embeddings)
    print("embedded and saved:", embeddings.shape)

loaded cached embeddings: (1478, 1536)


## Scoping the training set

The classifier is trained on focal clips, but only the species that are actually heard in the soundscapes can be evaluated against field truth. Training on species that never appear in the field data adds no measurable value, so the focal training set is scoped to the species that are both heard in the field and have focal audio available.

In [10]:
import pandas as pd

train = pd.read_csv('../data/train.csv')
tax = pd.read_csv('../data/taxonomy.csv')

# species heard in the field (from the soundscape labels)
heard = set(s for lst in labels for s in lst)

# of those, which have focal training audio
trainable = set(train['primary_label'].astype(str))
target_species = sorted(heard & trainable)
print("species to train on:", len(target_species))

# how many focal clips each target species has
counts = train[train['primary_label'].astype(str).isin(target_species)]['primary_label'].value_counts()
print("clips available per target species:")
print("  min:", counts.min(), " median:", int(counts.median()), " max:", counts.max())
print("  total clips if uncapped:", counts.sum())
print("  total if capped at 40 each:", int(np.minimum(counts, 40).sum()))

species to train on: 47
clips available per target species:
  min: 1  median: 77  max: 493
  total clips if uncapped: 6540
  total if capped at 40 each: 1445


## Selecting the clips to download

For each of the 47 target species, up to 40 focal clips are selected. Species with fewer than 40 keep all they have. This produces the exact list of files to download, so nothing beyond the training set is pulled.

In [ ]:
CAP = 40

# select up to CAP clips per target species
selected = (
    train[train['primary_label'].astype(str).isin(target_species)]
    .groupby('primary_label', group_keys=False)
    .apply(lambda g: g.head(CAP))
    .reset_index(drop=True)
)

print("clips selected:", len(selected))
print("species covered:", selected['primary_label'].nunique())
print(selected[['primary_label', 'filename']].head())

clips selected: 1445
species covered: 47
  primary_label                filename
0        116570  116570/iNat1460166.ogg
1         22961    22961/iNat846512.ogg
2         22961   22961/iNat1662496.ogg
3         22961   22961/iNat1657948.ogg
4         22961    22961/iNat839121.ogg


C:\Users\guech\AppData\Local\Temp\ipykernel_7000\1476392353.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.head(CAP))


## Downloading the focal clips

Each selected clip is pulled from the competition's train_audio folder into a local focal_audio directory. Files already present are skipped, so the download can be resumed if interrupted. This is roughly 1,445 small files, about a gigabyte.

In [ ]:
import os

FOCAL_DIR = '../data/focal_audio'
os.makedirs(FOCAL_DIR, exist_ok=True)

# check which files we already have locally and which we need to download
to_get = []
for fn in selected['filename']:
    local = os.path.join(FOCAL_DIR, fn)
    if not os.path.exists(local):
        to_get.append(fn)

print("already have:", len(selected) - len(to_get))
print("to download:", len(to_get))

already have: 0
to download: 1445


In [2]:
import os
import pandas as pd

FOCAL_DIR = '../data/focal_audio'
train = pd.read_csv('../data/train.csv')

# rebuild the 47 target species: heard in the field AND has focal audio
with open('../data/full_meta.json') as f:
    import json
    heard = set(s for lst in json.load(f)['labels'] for s in lst)
trainable = set(train['primary_label'].astype(str))
target_species = sorted(heard & trainable)

species_folders = [str(sp) for sp in target_species]
print("species folders to pull:", len(species_folders))

species folders to pull: 47


In [ ]:
import os, json
import pandas as pd
import kagglehub

FOCAL_DIR = '../data/focal_audio'
train = pd.read_csv('../data/train.csv')

# rebuild the 47 target species: heard in the field AND has focal audio
with open('../data/full_meta.json') as f:
    heard = set(s for lst in json.load(f)['labels'] for s in lst)
trainable = set(train['primary_label'].astype(str))
target_species = sorted(heard & trainable)

CAP = 40
selected = (
    train[train['primary_label'].astype(str).isin(target_species)]
    .groupby('primary_label', group_keys=False)
    .apply(lambda g: g.head(CAP), include_groups=False)
    .reset_index(drop=True)
)
# apply put filename back; rebuild it cleanly
selected = (
    train[train['primary_label'].astype(str).isin(target_species)]
    .groupby('primary_label', group_keys=False)
    .head(CAP)
    .reset_index(drop=True)
)
print("clips to fetch:", len(selected))

clips to fetch: 1445


In [9]:
import numpy as np, json
fe = np.load('../data/focal_embeddings.npy')
with open('../data/focal_labels.json') as f:
    fl = json.load(f)
print("focal embeddings:", fe.shape)
print("focal labels:", len(fl), "| species:", len(set(fl)))

focal embeddings: (1445, 1536)
focal labels: 1445 | species: 47


In [ ]:
!pip install -q git+https://github.com/google-research/perch-hoplite.git

import os, json
import numpy as np
import pandas as pd
import librosa
from perch_hoplite.zoo import model_configs

BASE = '/kaggle/input/birdclef-2026'
SR = 32000
CAP = 40

# load the training data and the soundscape labels
train = pd.read_csv(f'{BASE}/train.csv')
ss = pd.read_csv(f'{BASE}/train_soundscapes_labels.csv')

heard = set(s for v in ss['primary_label'].astype(str) for s in v.split(';')) # species heard in the field
trainable = set(train['primary_label'].astype(str))
target = sorted(heard & trainable)

selected = (train[train['primary_label'].astype(str).isin(target)] # select up to CAP clips per target species
            .groupby('primary_label', group_keys=False).head(CAP).reset_index(drop=True)) 
print("clips:", len(selected), "species:", selected['primary_label'].nunique())

model = model_configs.load_model_by_name('perch_v2') # load the Perch model

# embed the selected focal audio clips and save the embeddings and labels
emb, labs, missing = [], [], 0
for n, (fn, lab) in enumerate(zip(selected['filename'], selected['primary_label'].astype(str)), 1):
    path = f'{BASE}/train_audio/{fn}'
    if not os.path.exists(path):
        missing += 1; continue
    y, _ = librosa.load(path, sr=SR)
    target_len = 5 * SR
    y = np.pad(y, (0, target_len - len(y))) if len(y) < target_len else y[:target_len]
    out = model.embed(y.astype(np.float32))
    emb.append(np.asarray(out.embeddings).reshape(-1))
    labs.append(lab)
    if n % 200 == 0: print(f"{n}/{len(selected)}")

emb = np.array(emb)
np.save('/kaggle/working/focal_embeddings.npy', emb)
with open('/kaggle/working/focal_labels.json', 'w') as f:
    json.dump(labs, f)
print("saved:", emb.shape, "missing:", missing)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/birdclef-2026/train.csv'

In [ ]:
p = kagglehub.competition_download('birdclef-2026', path=f"train_audio/{selected['filename'].iloc[0]}")
print("got:", p)
print("exists:", os.path.exists(p)) # check that the file was downloaded successfully

KaggleApiHTTPError: 404 Client Error.

Resource not found at URL: https://www.kaggle.com/competitions/birdclef-2026
Please make sure you specified the correct resource identifiers.

In [ ]:
import os

# check which species folders we need to pull from the competition dataset
species_folders = sorted(set(os.path.dirname(fn) for fn in selected['filename']))
print("species folders to pull:", len(species_folders))

done = 0
for sp in species_folders:
    dest = os.path.join(FOCAL_DIR, sp)
    os.makedirs(dest, exist_ok=True)
    !python -m kaggle competitions download -c birdclef-2026 -f train_audio/{sp} -p {dest} --force -q
    done += 1
    print(f"{done}/{len(species_folders)} folders")

NameError: name 'selected' is not defined

In [ ]:
import os

for n, fn in enumerate(to_get, 1):
    dest = os.path.join(FOCAL_DIR, os.path.dirname(fn))
    os.makedirs(dest, exist_ok=True)
    !python -m kaggle competitions download -c birdclef-2026 -f train_audio/{fn} -p {dest} --force -q
    if n % 100 == 0:
        print(f"{n}/{len(to_get)} downloaded")

print("done")

100/1445 downloaded
^C
200/1445 downloaded


## Embedding the focal clips

Each downloaded focal clip is loaded, trimmed or padded to 5 seconds to match Perch's input, and embedded the same way as the soundscapes. Focal clips vary in length, so a fixed 5-second window is taken from the start. The result is one 1536-dimensional vector per clip, paired with its species label.

In [ ]:
# embed the selected focal audio clips and save the embeddings and labels
if os.path.exists('../data/focal_embeddings.npy'):
    focal_embeddings = np.load('../data/focal_embeddings.npy')
    with open('../data/focal_labels.json') as f:
        focal_labels = json.load(f)
    print("loaded cached focal embeddings:", focal_embeddings.shape)
else:
    focal_embeddings = []
    focal_labels = []
    missing = 0
    for n, (fn, lab) in enumerate(zip(selected['filename'], selected['primary_label'].astype(str)), 1):
        path = os.path.join(FOCAL_DIR, fn)
        if not os.path.exists(path):
            missing += 1
            continue
        y, _ = librosa.load(path, sr=SR)
        target = 5 * SR
        if len(y) < target:
            y = np.pad(y, (0, target - len(y)))
        else:
            y = y[:target]
        out = model.embed(y.astype(np.float32))
        focal_embeddings.append(np.asarray(out.embeddings).reshape(-1))
        focal_labels.append(lab)
        if n % 200 == 0:
            print(f"{n}/{len(selected)} embedded")
    focal_embeddings = np.array(focal_embeddings)
    with open('../data/focal_labels.json', 'w') as f:
        json.dump(focal_labels, f)
    np.save('../data/focal_embeddings.npy', focal_embeddings)
    print("embedded and saved:", focal_embeddings.shape, "missing:", missing)

loaded cached focal embeddings: (0,)


In [ ]:
# clear the bad cache if it exists
import os
os.remove('../data/focal_embeddings.npy')
if os.path.exists('../data/focal_labels.json'):
    os.remove('../data/focal_labels.json')
print("cleared bad cache")

cleared bad cache


In [ ]:
# check how many focal clips are on disk
import glob
found = glob.glob('../data/focal_audio/**/*.ogg', recursive=True)
print("focal clips on disk:", len(found))

focal clips on disk: 0


## Notebook summary

This notebook is the working record of getting the BirdCLEF audio out of Kaggle and into a usable form. It is deliberately left messy, since it documents the obstacles and how each was resolved rather than presenting a clean final pipeline. The clean, reproducible work continues in notebook 09.

The main problems, and how they were solved:

**Per-file downloads stopped working.** Both the Kaggle CLI and kagglehub returned 404 errors when fetching individual audio files, and the CLI folder download was rejected because the flag only accepts single files. Downloading clips one at a time also proved far too slow, since each call re-authenticated from scratch, taking roughly twenty minutes per hundred files.

**Windows long-path limits broke installation.** Perch depends on TensorFlow, and installing it failed because the project sat inside a deeply nested OneDrive folder that pushed file paths past the Windows 260-character limit. This was fixed by moving the project to a short path (C:\dev) and rebuilding the virtual environment in place, since a moved environment keeps its old hardcoded paths and breaks.

**The resolution: embed on Kaggle, analyse locally.** Rather than pull tens of gigabytes of audio down, the Perch embedding was run once on Kaggle, where the data is already mounted and a GPU is available. Only the small embedding files were downloaded. A path detail caught this out too: the competition data mounts under /kaggle/input/competitions/birdclef-2026, one level deeper than expected.

The outcome is two saved embedding files, the focal clips for training and the soundscape segments for evaluation, which all later notebooks load directly. No audio or heavy computation is repeated after this point.